# **CatBoost Model Development and Evaluation Script**

---

## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Optuna`***: Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***`RandomizedSearchCV`***: Sklearn-compatible baseline optimization for fair comparison and reproducibility.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***CatBoost (Baseline)***
    - ***CatBoost (Optuna)***
    - ***CatBoost (RandomizedSearchCV)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

## Import Libraries and Root Configuration

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"

In [3]:
# records and calculations
import pandas as pd
import numpy as np

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')

# visualizations
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, randint, loguniform

# gradient boosting model
import catboost as cb

# optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# import helper functions
from src.utilities import Evaluator, DataHandler, ModelPersister, CatBoostPruningCallback

## Load Dataset and Artifacts

In [4]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [5]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010,-0.002307


In [6]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.000679,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,-0.004875,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,0.060478,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,-0.058245,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,0.009339,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010


In [7]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [8]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

In [9]:
# models to be applied
MODELS = ["CatBoost (Baseline)", "CatBoost (Optuna)", "CatBoost (RandomSearchCV)"]

In [10]:
# Configuration
TRANSACTION_COST = 0.0005
RISK_FREE_RATE = 0.02/252
EARLY_STOPPING_ROUNDS = 50
N_TRIALS = 50

# **Baseline Modeling**

In [11]:
# artifacts for catboost
x_train_cb, x_test_cb, cat_features, cat_indices = DataHandler.prepare_for_catboost(x_train, x_test)

print(f"categorical features: {cat_features}")
print(f"cat indices: {cat_indices}")

categorical features: []
cat indices: None


In [12]:
# baseline model parameters
BASELINE_PARAMS = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'random_strength': 1,
    'bagging_temperature': 1,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': False
}

In [13]:
# training the baseline model
baseline_model = cb.CatBoostRegressor(**BASELINE_PARAMS)
baseline_model.fit(x_train_cb, y_train,
                   eval_set=(x_test_cb, y_test),
                   early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                   verbose=False
                )

In [14]:
print(f"Best iteration: {baseline_model.best_iteration_}")
print(f"Trees actually built: {baseline_model.tree_count_}")

Best iteration: 25
Trees actually built: 26


##  Apply Model to Make Prediction

In [15]:
# baseline prediction on train and test set
y_train_pred_baseline = baseline_model.predict(x_train_cb)
y_test_pred_baseline = baseline_model.predict(x_test_cb)

## Evaluating Model Performance
Calculating Errors for train and test data

In [16]:
# evaluation of train metrics
train_metrics_baseline = Evaluator.calculate_metrics(y_train, y_train_pred_baseline)

# evaluation of test metrics
test_metrics_baseline = Evaluator.calculate_metrics(y_test, y_test_pred_baseline)

In [17]:
# calculate directional time-series specific accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test, y_test_pred_baseline)

In [18]:
# calcluate financial metrics of test data for baseline model
fin_baseline = Evaluator.financial_metrics('CatBoost (Baseline)', y_test, y_test_pred_baseline)

display(fin_baseline)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,CatBoost (Baseline),0.0187,0.0347,-14.1984,2.6074


In [19]:
# baseline model performance
baseline_perf = Evaluator.performance_table(train_metrics_baseline + [train_acc_baseline], test_metrics_baseline + [test_acc_baseline])

print("CatBoost - Baseline Modeling Performance")
display(baseline_perf)

CatBoost - Baseline Modeling Performance


,Metrics,Training,Test
0,MSE,0.0002,0.0001
1,MAE,0.0093,0.0060
2,RMSE,0.0133,0.0081
3,R2 Score,0.0863,0.0024
4,MAPE,3769.6861,8865.7085
5,Directional Accuracy (%),60.2957,49.6718


# **Optuna Hyperparameter Optimzation**

## Find Best Parameters

In [20]:
# define objective function for optuna optimization
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 50),
        'border_count': trial.suggest_categorical('border_count', [64, 128, 254]),
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'task_type': 'CPU',
        'verbose': False
    }
    
    sharpe_scores = []
    rmse_scores = []
    dir_acc_scores = []

    for _, (train_idx, val_idx) in enumerate(cv.split(x_train_cb)):
        x_tr, x_val = x_train_cb.iloc[train_idx], x_train_cb.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # create fresh model each fold
        model = cb.CatBoostRegressor(**params)

        # pruning callback for early trial termination
        pruning_cb = CatBoostPruningCallback(trial)

        # early stopping inside each fold
        model.fit(x_tr, y_tr,
                  eval_set=(x_val, y_val),
                  early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                  verbose=False,
                  callbacks=[pruning_cb]
                 )
        
        y_pred = model.predict(x_val)

        # calculate rmse
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_scores.append(rmse)

        # calculate sharpe ratio with transaction costs
        positions = np.sign(y_pred)
        y_val_true = y_val.values

        # apply transaction costs based on position changes
        position_changes = np.abs(np.diff(positions, prepend=0))
        costs = position_changes * TRANSACTION_COST

        # strategy returns after costs
        strategy_returns = positions * y_val_true - costs
        strategy_returns = strategy_returns[~np.isnan(strategy_returns)]            # filter out NaN/zero-variance

        if len(strategy_returns) > 0 and np.std(strategy_returns) > 0:
            excess_returns = strategy_returns - RISK_FREE_RATE
            sharpe = np.mean(excess_returns) / np.std(strategy_returns) * np.sqrt(252)
            sharpe_scores.append(sharpe)
        
        else:
            sharpe_scores.append(0.0)
        
        # calculate directional accuracy
        true_direction = np.sign(y_val_true)
        pred_direction = np.sign(y_pred)

        dir_acc = np.mean(true_direction == pred_direction) * 100
        dir_acc_scores.append(dir_acc)
    
    # aggregate metrics (mean values)
    mean_sharpe = np.mean(sharpe_scores)
    mean_rmse = np.mean(rmse_scores)
    mean_dir_acc = np.mean(dir_acc_scores)
    
    # store metrics as user attributes for logging
    trial.set_user_attr("sharpe_ratio", mean_sharpe)
    trial.set_user_attr("rmse", mean_rmse)
    trial.set_user_attr("directional_accuracy", mean_dir_acc)

    SHARPE_WEIGHT = 2.0
    RMSE_WEIGHT = 1.0
    DIR_ACC_WEIGHT = 1.0

    y_mean = np.mean(y_train.values)
    norm_rmse = mean_rmse / y_mean

    norm_sharpe = np.clip(mean_sharpe, -5, 5)
    norm_sharpe = (norm_sharpe + 5) / 10
    norm_sharpe = 1 - norm_sharpe

    norm_dir_acc = 1 - (mean_dir_acc / 100)

    final_score = SHARPE_WEIGHT * norm_sharpe + RMSE_WEIGHT * norm_rmse + DIR_ACC_WEIGHT * norm_dir_acc

    return final_score

In [21]:
# enable pruning to skip the bad trials early
pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=20)

# create study object
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42), pruner=pruner)

In [22]:
# optimization
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# best parameter from optuna
best_params_optuna = study.best_params

best_params_optuna.update({
    'eval_metric': 'RMSE',
    'loss_function': 'RMSE',
    'random_seed': 42,
    'verbose': False
})

Best trial: 40. Best value: 47.7625: 100%|██████████| 50/50 [15:58<00:00, 19.18s/it]


In [23]:
# train final model with best parameters
optuna_model = cb.CatBoostRegressor(**best_params_optuna)

optuna_model.fit(x_train_cb, y_train,
                 eval_set=(x_test_cb, y_test),
                 early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                 verbose=False
                )

In [24]:
print(f"CatBoost best iteration: {optuna_model.best_iteration_}")
print(f"Trees built: {optuna_model.tree_count_}")

CatBoost best iteration: 20
Trees built: 21


## Apply model to make prediction

In [25]:
# apply model to make predictions on train and test set
y_train_pred_optuna = optuna_model.predict(x_train)
y_test_pred_optuna = optuna_model.predict(x_test)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [26]:
""" calculate train and test metrics
return [mse, mae, rmse, r2, mape] """
train_metrics_optuna = Evaluator.calculate_metrics(y_train, y_train_pred_optuna)
test_metrics_optuna = Evaluator.calculate_metrics(y_test, y_test_pred_optuna)

In [27]:
# calculate directional time-series specific accuracy
train_acc_optuna = Evaluator.directional_accuracy(y_train, y_train_pred_optuna)
test_acc_optuna = Evaluator.directional_accuracy(y_test, y_test_pred_optuna)

In [28]:
# calcluate financial metrics of test data for Optuna model
fin_optuna = Evaluator.financial_metrics('CatBoost (Optuna)', y_test, y_test_pred_optuna)

display(fin_optuna)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,CatBoost (Optuna),0.3632,0.6086,-10.9739,11.1651


In [29]:
# performance table of optuna optimization
optuna_perf = Evaluator.performance_table(train_metrics_optuna + [train_acc_optuna], test_metrics_optuna + [test_acc_optuna])

print("CatBoost - optuna Modeling Performance")
display(optuna_perf)

CatBoost - optuna Modeling Performance


,Metrics,Training,Test
0,MSE,0.0002,0.0001
1,MAE,0.0089,0.0060
2,RMSE,0.0125,0.0081
3,R2 Score,0.1819,0.0115
4,MAPE,3187.4633,11508.0825
5,Directional Accuracy (%),66.7032,52.2976


#  RandomizedSearchCV Optimzation

## Train RandomizedSearchCV Model

In [30]:
BASELINE_PARAMS = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': False,
}

# parameter distribution
RANDOM_PARAMS = {
    "iterations": randint(200, 1000),
    "learning_rate": loguniform(0.01, 0.3),
    "depth": randint(3, 13),
    "l2_leaf_reg": loguniform(0.1, 10.0),
    "subsample": uniform(0.5, 1),
    "bagging_temperature": loguniform(0.1, 5.0),
    "random_strength": loguniform(0.1, 5.0)
}

In [31]:
# catboost base model
model = cb.CatBoostRegressor(**BASELINE_PARAMS)

In [32]:
# optimize with RandomSearchCV
random_search = RandomizedSearchCV(model,
                                   RANDOM_PARAMS,
                                   n_iter=N_TRIALS,
                                   scoring='neg_root_mean_squared_error',
                                   cv=cv,
                                   random_state=42,
                                   refit=True,
                                   verbose=0,
                                   n_jobs=-1)

# fit model
random_search.fit(x_train_cb, y_train,
                   eval_set=(x_test_cb, y_test),
                   early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                   verbose=False
                )

,estimator,<catboost.cor...00297DEBF6EC0>
,param_distributions,"{'bagging_temperature': <scipy.stats....00297DEBF6BF0>, 'depth': <scipy.stats....00297DEBF7E80>, 'iterations': <scipy.stats....00297DEAD2CE0>, 'l2_leaf_reg': <scipy.stats....00297DEBF5450>, ...}"
,n_iter,50
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,TimeSeriesSpl...est_size=None)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [33]:
# best estimator from the RandomsSearchCV
randomSearch_model = random_search.best_estimator_

In [34]:
print(f"Best Score  : {random_search.best_score_:.4f}")
random_search.best_params_

Best Score  : -0.0117


{'bagging_temperature': 0.10020378002757138,
 'depth': 11,
 'iterations': 458,
 'l2_leaf_reg': 0.10325337616482036,
 'learning_rate': 0.05681142678077596,
 'random_strength': 0.5118807219845012,
 'subsample': 0.7221078104707302}

## Apply Model to Make Predictions

In [35]:
# make predictions on train and test set
y_train_pred_random = randomSearch_model.predict(x_train_cb)
y_test_pred_random = randomSearch_model.predict(x_test_cb)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [36]:
# calculate train and test metrics
train_metrics_random = Evaluator.calculate_metrics(y_train, y_train_pred_random)
test_metrics_random = Evaluator.calculate_metrics(y_test, y_test_pred_random)

In [37]:
# calculate directional time-series specific accuracy
train_acc_random = Evaluator.directional_accuracy(y_train, y_train_pred_random)
test_acc_random = Evaluator.directional_accuracy(y_test, y_test_pred_random)

In [38]:
# calcluate financial metrics of test data for RandomSearch model
fin_random = Evaluator.financial_metrics('CatBoost (RandomSearchCV)', y_test, y_test_pred_random)

display(fin_random)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,CatBoost (RandomSearchCV),-0.8286,-1.3101,-20.8284,-15.9136


In [39]:
# performance table of randomsearchCV optimization
random_perf = Evaluator.performance_table(train_metrics_random + [train_acc_random], test_metrics_random + [test_acc_random])

print("CatBoost - RandomSearchCV Optimized Modeling Performance")
display(random_perf)

CatBoost - RandomSearchCV Optimized Modeling Performance


,Metrics,Training,Test
0,MSE,0.0002,0.0001
1,MAE,0.0093,0.0060
2,RMSE,0.0134,0.0081
3,R2 Score,0.0719,0.0014
4,MAPE,2943.3705,9643.8462
5,Directional Accuracy (%),59.0361,49.4530


# Cross Valiation (All Models)

In [40]:
# cross validation of all models
cv_baseline = Evaluator.cv_evaluate(baseline_model, x_train_cb, y_train, cv)
cv_optuna = Evaluator.cv_evaluate(optuna_model, x_train_cb, y_train, cv)
cv_random = Evaluator.cv_evaluate(randomSearch_model, x_train_cb, y_train, cv)

In [41]:
# Evaluate cross-validation results
cv_df = pd.DataFrame({
    "Model": MODELS,
    "CV MSE": [cv_baseline['CV MSE'], cv_optuna['CV MSE'], cv_random['CV MSE']],
    "CV MAE": [cv_baseline['CV MAE'], cv_optuna['CV MAE'], cv_random['CV MAE']],
    "CV RMSE": [cv_baseline['CV RMSE'], cv_optuna['CV RMSE'], cv_random['CV RMSE']],
    "CV R2": [cv_baseline['CV R2'], cv_optuna['CV R2'], cv_random['CV R2']],
    "CV MAPE": [cv_baseline['CV MAPE'], cv_optuna['CV MAPE'], cv_random['CV MAPE']],
    "CV Directional Accuracy (%)": [cv_baseline['CV Directional Accuracy (%)'], cv_optuna['CV Directional Accuracy (%)'], cv_random['CV Directional Accuracy (%)']]
}).round(4)

print("Cross-Validation Results:")
display(cv_df)

Cross-Validation Results:


,Model,CV MSE,CV MAE,CV RMSE,CV R2,CV MAPE,CV Directional Accuracy (%)
0,CatBoost (Baseline),0.0002,0.0091,0.0127,-0.1679,63035.5723,50.5263
1,CatBoost (Optuna),0.0002,0.0088,0.0123,-0.0958,84317.3790,51.3158
2,CatBoost (RandomSearchCV),0.0001,0.0087,0.0122,-0.0686,51041.2248,51.9079


# Summarize Models Performance

In [42]:
# test metrics of all models
test_metrics = [test_metrics_baseline, test_metrics_optuna, test_metrics_random]

# test directional accuracy of all models
test_dir_acc = [test_acc_baseline, test_acc_optuna, test_acc_random]

# performance summary including cross validation results
performance_summary = Evaluator.summary_builder(MODELS, cv_df, test_metrics, test_dir_acc)

In [43]:
# Display the final performance summary table
print("Overall Model Performance:")
display(performance_summary)

Overall Model Performance:


,Model,CV MSE,CV MAE,CV RMSE,CV R2,CV MAPE,CV Directional Accuracy (%),Test MSE,Test MAE,Test RMSE,Test R2,Test MAPE,Test Directional Accuracy (%)
0,CatBoost (Baseline),0.0002,0.0091,0.0127,-0.1679,63035.5723,50.5263,0.0001,0.006,0.0081,0.0024,8865.7085,49.6718
1,CatBoost (Optuna),0.0002,0.0088,0.0123,-0.0958,84317.3790,51.3158,0.0001,0.006,0.0081,0.0115,11508.0825,52.2976
2,CatBoost (RandomSearchCV),0.0001,0.0087,0.0122,-0.0686,51041.2248,51.9079,0.0001,0.006,0.0081,0.0014,9643.8462,49.4530


# Overfitting Analysis

In [44]:
# Overfitting analysis of all models

rows = []
idx_count = 0
for _, row in performance_summary.iterrows():
    # Extract metrics needed for assess_overfitting
    cv_r2 = row['CV R2']
    test_r2 = row['Test R2']
    cv_rmse = row['CV RMSE']
    test_rmse = row['Test RMSE']
    
    # get overfitting and generalization status
    gap, rmse_ratio, overfit_status, gen_status = Evaluator.assess_overfitting(
        cv_r2=cv_r2,
        test_r2=test_r2,
        cv_rmse=cv_rmse,
        test_rmse=test_rmse,
        tolerance=0.05
    )

    # directional accuracy (supplementary)
    cv_dir_acc = row.get('CV Directional Accuracy (%)', 0) 
    test_dir_acc = [test_acc_baseline, test_acc_optuna, test_acc_random][idx_count]
    dir_acc_gap = test_dir_acc - cv_dir_acc
    
    # build row
    rows.append({
        "Model": row['Model'],
        "CV RMSE": row['CV RMSE'],
        "CV R2": row['CV R2'],
        "CV Dir Acc (%)": cv_dir_acc,
        "Test RMSE": row['Test RMSE'],
        "Test R2": row['Test R2'],
        "Test Dir Acc (%)": test_dir_acc,
        "R2 Gap": gap,
        "RMSE Ratio": rmse_ratio,
        "Dir Acc Gap (%)": dir_acc_gap,
        "Overfitting Status": overfit_status,
        "Model Status (Generalization)": gen_status
    })

    idx_count += 1

overfit_df = pd.DataFrame(rows).round(4)

In [45]:
print("CatBoost - Overfitting Analysis:")
display(overfit_df)

CatBoost - Overfitting Analysis:


,Model,CV RMSE,CV R2,CV Dir Acc (%),Test RMSE,Test R2,Test Dir Acc (%),R2 Gap,RMSE Ratio,Dir Acc Gap (%),Overfitting Status,Model Status (Generalization)
0,CatBoost (Baseline),0.0127,-0.1679,50.5263,0.0081,0.0024,49.6718,-0.1703,0.6378,-0.8545,Mild,Poor
1,CatBoost (Optuna),0.0123,-0.0958,51.3158,0.0081,0.0115,52.2976,-0.1073,0.6585,0.9818,Mild,Poor
2,CatBoost (RandomSearchCV),0.0122,-0.0686,51.9079,0.0081,0.0014,49.4530,-0.0700,0.6639,-2.4549,Mild,Poor
